# VDOS y participation ratio (PR) desde una MD

Como en Gürbüz y Sanyal, arXiv:2305.08134 (Ecs. 7 y 8).

- **VDOS por átomo**: $\mathrm{VDOS}_i(\nu) = m_i \sum_{\alpha=x,y,z} |v_{i\alpha}(\nu)|^2$, espectro de potencia de las velocidades (equivale a la transformada de Fourier de la autocorrelación de velocidades de la Ec. 7).
- **VDOS total**: $g(\nu) = \sum_i \mathrm{VDOS}_i(\nu)$, normalizada a $\int g\,d\nu = 1$ (THz⁻¹).
- **Contribuciones parciales** (por elemento, y en el plano / fuera del plano): misma normalización que la total, así que **suman la total** y nunca la superan. El área de cada elemento es $N_\mathrm{elem}/N$; la del plano ≈ 2/3 y la de fuera ≈ 1/3.
- **PR total** (Ec. 8): $\mathrm{PR}(\nu) = \dfrac{(\sum_i \mathrm{VDOS}_i^q)^2}{N\sum_i \mathrm{VDOS}_i^{2q}}$, con $q = 2$ como en el artículo.

Ojo al leer la PR: también baja sin que haya localización cuando un modo solo mueve una subred (p. ej. solo los 18 B de 126 átomos → PR ≈ 0.14) y por el ruido estadístico de una MD finita (≈ 0.9 en lugar de 1).

## 1. Parámetros

In [ ]:
TRAYECTORIA = "md_bgdy3x3_2000_2fs.traj"  # .traj de ASE (con velocidades) o extxyz con posiciones y time_fs
DT_FS = 2.0        # tiempo entre frames guardados (fs); None = de time_fs o estimado de posiciones y velocidades
PASO = 1            # leer 1 de cada PASO frames
INICIO_PS = 1.0     # descarta el equilibrado
FIN_PS = None       # None = hasta el final
SEG_PS = 2.0        # segmento de Welch: resolución ≈ 1.5/SEG_PS THz (más largo = más resolución y más ruido)
Q = 2               # exponente de la Ec. 8 (2 como en el artículo; 1 = PR de vectores propios)
CON_MASA = True     # ponderar por masa (la normalización 1/3NkBT de la Ec. 7 lo exige)
SOMBREAR = True     # gris donde g < 2 % del máximo: allí no hay modos y la PR no significa nada

## 2. Código

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from ase import units
from ase.io import iread
from ase.geometry import find_mic
from scipy.signal import welch

KB = 8.617333262e-5     # eV/K
THZ_A_CM1 = 33.35641    # 1 THz = 33.356 cm⁻¹


def leer_trayectoria(ruta, paso=1, dt_fs=None):
    """Posiciones, velocidades, masas, símbolos y tiempo entre frames (fs).
    Si no hay velocidades, se sacan de las posiciones: v(t+Δ/2) = [x(t+Δ) − x(t)]/Δ."""
    X, V, tiempos, at0 = [], [], [], None
    for at in iread(ruta, index=f"::{paso}"):
        if at0 is None:
            at0 = at
        X.append(at.get_positions())
        if at.has("momenta"):
            V.append(at.get_velocities())          # unidades de ASE: m·v² sale en eV
        if "time_fs" in at.info:
            tiempos.append(at.info["time_fs"])
    X = np.array(X)
    V = np.array(V) if len(V) == len(X) else None

    def mic(d):                                    # desplazamientos con imagen mínima
        v, _ = find_mic(d.reshape(-1, 3), at0.cell, pbc=at0.pbc)
        return v.reshape(d.shape)

    if dt_fs is not None:
        origen = "dado"
    elif len(tiempos) == len(X):
        dt_fs, origen = float(np.median(np.diff(tiempos))), "time_fs"
    elif V is not None:
        # Verlet: x(t+Δ) − x(t) = v(t)Δ + a(t)Δ²/2 y <a·v> = 0 → ajuste por mínimos cuadrados.
        # Exacto si los frames son pasos consecutivos; si se guardó cada k pasos, da DT_FS a mano.
        n = min(len(X) - 1, 2000)
        dx, v0 = mic(X[1:n + 1] - X[:n]), V[:n]
        dt_fs, origen = float(np.sum(dx * v0) / np.sum(v0 * v0) / units.fs), "estimado de x y v"
    else:
        raise ValueError("Sin velocidades ni time_fs: indica DT_FS.")

    por_diferencias = V is None
    if por_diferencias:
        V = mic(X[1:] - X[:-1]) / (dt_fs * units.fs)
    return dict(V=V, masas=at0.get_masses(), simbolos=np.array(at0.get_chemical_symbols()),
                dt_fs=dt_fs, origen_dt=origen, por_diferencias=por_diferencias, ruta=ruta)


def espectros_por_atomo(V, pesos, dt_fs, seg_ps):
    """Welch (Hann, 50 % de solape) de las velocidades de cada átomo y componente.
    Devuelve ν (THz) y S (N, 3, nν) con ∫ S dν = <peso·v²> (en eV/THz si los pesos son masas)."""
    M, N, _ = V.shape
    L = min(M, int(round(seg_ps * 1000 / dt_fs)))
    S = []
    for a in range(0, N, 64):                     # por bloques de átomos (memoria)
        nu, P = welch(V[:, a:a + 64], fs=1000 / dt_fs, window="hann", nperseg=L,
                      noverlap=L // 2, detrend=False, axis=0)
        S.append(P * pesos[a:a + 64, None])
    return nu, np.concatenate(S, axis=1).transpose(1, 2, 0), L


def participation_ratio(G, q=2):
    """Ec. 8: (Σ_i G_i^q)² / (N Σ_i G_i^2q), G = VDOS_i (n_átomos, nν)."""
    G = G / G.max()
    with np.errstate(invalid="ignore", divide="ignore"):
        return (G ** q).sum(0) ** 2 / (G.shape[0] * (G ** (2 * q)).sum(0))


def calcular(d, inicio_ps=1.0, fin_ps=None, seg_ps=2.0, q=2, con_masa=True):
    V, m, sim, dt = d["V"], d["masas"], d["simbolos"], d["dt_fs"]
    N = len(m)
    t = np.arange(len(V)) * dt / 1000 + (dt / 2000 if d["por_diferencias"] else 0)
    T_inst = np.einsum("i,tij->t", m, V ** 2) / (3 * N * KB)
    sel = (t >= inicio_ps) & (t <= (fin_ps if fin_ps is not None else np.inf))
    Vs = V[sel] - (np.einsum("i,tij->tj", m, V[sel]) / m.sum())[:, None, :]   # sin centro de masas
    pesos = m if con_masa else np.ones(N)

    def vdos(Vtramo):
        nu, S, L = espectros_por_atomo(Vtramo, pesos, dt, seg_ps)
        if d["por_diferencias"]:                   # respuesta de la diferencia finita: sinc(νΔ)
            S = S / np.maximum(np.sinc(nu * dt / 1000) ** 2, 1e-3)
        return nu, S, L

    nu, S, L = vdos(Vs)
    dnu = nu[1] - nu[0]
    area = S.sum() * dnu                            # normalización común: todo suma la total
    g = S.sum((0, 1)) / area
    r = dict(nu=nu, g=g, dnu=dnu, N=N, M=len(Vs), dt_fs=dt, L=L, q=q,
             resolucion=1.5 * 1000 / (L * dt), nyquist=nu[-1],
             g_plano=S[:, :2].sum((0, 1)) / area, g_fuera=S[:, 2].sum(0) / area,
             g_elem={e: S[sim == e].sum((0, 1)) / area for e in sorted(set(sim))},
             T_elem={e: S[sim == e].sum() * dnu / (3 * (sim == e).sum() * KB) for e in sorted(set(sim))},
             pr=participation_ratio(S.sum(1), q),
             t=t, T_inst=T_inst, sel=sel, T_media=T_inst[sel].mean(), datos=d)
    r["mitades"] = []                               # VDOS de cada mitad, para ver si es estacionaria
    for h in (Vs[: len(Vs) // 2], Vs[len(Vs) // 2:]):
        if len(h) >= L:
            Sh = vdos(h)[1].sum((0, 1))
            r["mitades"].append(Sh / (Sh.sum() * dnu))
    cum = np.cumsum(g) * dnu
    r["nu_max"] = min(nu[np.searchsorted(cum, 0.995)] * 1.1, nu[-1])
    return r


def resumen(r):
    d = r["datos"]
    print(f"{d['ruta']}: {r['N']} átomos, dt entre frames {r['dt_fs']:.4f} fs ({d['origen_dt']})"
          + ("; velocidades a partir de las posiciones" if d["por_diferencias"] else ""))
    print(f"Usados {r['M']} frames ({r['M'] * r['dt_fs'] / 1000:.2f} ps); T media {r['T_media']:.0f} K")
    print(f"Segmentos de {r['L']} frames ({r['L'] * r['dt_fs'] / 1000:.2f} ps): resolución ≈ "
          f"{r['resolucion']:.2f} THz ({r['resolucion'] * THZ_A_CM1:.0f} cm⁻¹); Nyquist {r['nyquist']:.0f} THz")
    print("Área de cada contribución: " + ", ".join(f"{e} {a.sum() * r['dnu']:.3f}" for e, a in r["g_elem"].items())
          + f"; plano {r['g_plano'].sum() * r['dnu']:.3f}, fuera {r['g_fuera'].sum() * r['dnu']:.3f}")
    print("T por elemento (equiparto): " + ", ".join(f"{e} {T:.0f} K" for e, T in r["T_elem"].items()))
    alto = r["g"][r["nu"] > 0.9 * r["nyquist"]].sum() * r["dnu"]
    if alto > 0.005:
        print(f"AVISO: {100 * alto:.1f} % de la VDOS cerca de Nyquist: posible aliasing (frames muy espaciados).")


def figura_vdos_pr(r, sombrear=True):
    nu, xm = r["nu"], r["nu_max"]
    colores = ["#d1495b", "#2e86ab", "#edae49", "#66a182", "#8d6a9f"]
    fig, ax = plt.subplots(2, 2, figsize=(11, 7.5), sharex=True)
    (a1, a2), (a3, a4) = ax

    a1.plot(nu, r["g"], color="k", lw=1.6, label="Total")
    for (e, ge), c in zip(r["g_elem"].items(), colores):
        a1.plot(nu, ge, color=c, lw=1.2, label=e)
    a1.set_title("VDOS total y contribución de cada elemento")

    a2.plot(nu, r["g"], color="k", lw=1.6)
    a2.set_title("VDOS total")

    a3.plot(nu, r["g"], color="k", lw=1.6, label="Total")
    a3.plot(nu, r["g_plano"], color="#2e86ab", lw=1.2, label="en el plano (x, y)")
    a3.plot(nu, r["g_fuera"], color="#d1495b", lw=1.2, label="fuera del plano (z)")
    a3.set_title("Contribuciones en el plano y fuera del plano")

    for a in (a1, a2, a3):
        a.set_ylabel("g(ν) (1/THz)")
        a.set_ylim(bottom=0)
    for a in (a1, a3):
        a.legend(fontsize=8, frameon=False)

    if sombrear:
        a4.fill_between(nu, 0, 1, where=r["g"] < 0.02 * r["g"].max(), color="0.92", lw=0,
                        transform=a4.get_xaxis_transform(), label="sin modos (g < 2 %)")
        a4.legend(fontsize=8, frameon=False, loc="lower left")
    a4.plot(nu, r["pr"], color="k", lw=1.2)
    a4.set_ylim(0, 1.05)
    a4.set_ylabel("PR")
    a4.set_title(f"PR total, Ec. 8 (q = {r['q']:g})")

    for a in (a3, a4):
        a.set_xlabel("ν (THz)")
    for a in ax.ravel():
        a.set_xlim(0, xm)
    for a in (a1, a2):
        a.secondary_xaxis("top", functions=(lambda x: x * THZ_A_CM1, lambda x: x / THZ_A_CM1)).set_xlabel("cm⁻¹", fontsize=8)
    fig.suptitle(f"{r['datos']['ruta']}: {r['N']} átomos, T = {r['T_media']:.0f} K, "
                 f"{r['M'] * r['dt_fs'] / 1000:.1f} ps, resolución {r['resolucion']:.2f} THz", fontsize=10)
    fig.tight_layout()
    return fig


def figura_diagnostico(r):
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
    ax[0].plot(r["t"], r["T_inst"], lw=0.5, color="0.4")
    ax[0].axvspan(r["t"][0], r["t"][r["sel"]][0], color="#d1495b", alpha=0.15, label="descartado")
    ax[0].axhline(r["T_media"], color="k", ls="--", lw=0.8, label=f"media {r['T_media']:.0f} K")
    ax[0].set_xlabel("t (ps)")
    ax[0].set_ylabel("T instantánea (K)")
    ax[0].legend(fontsize=8, frameon=False)
    ax[1].plot(r["nu"], r["g"], color="k", lw=1.4, label="toda")
    for h, c, lab in zip(r["mitades"], ["#2e86ab", "#d1495b"], ["1.ª mitad", "2.ª mitad"]):
        ax[1].plot(r["nu"], h, color=c, lw=0.9, label=lab)
    ax[1].set_xlim(0, r["nu_max"])
    ax[1].set_ylim(bottom=0)
    ax[1].set_xlabel("ν (THz)")
    ax[1].set_ylabel("g(ν) (1/THz)")
    ax[1].set_title("VDOS de cada mitad (deben coincidir)", fontsize=9)
    ax[1].legend(fontsize=8, frameon=False)
    fig.tight_layout()
    return fig

## 3. Lectura

In [ ]:
# Se guarda en memoria: si solo cambias parámetros del cálculo, no se vuelve a leer el archivo
if globals().get("_clave") != (TRAYECTORIA, PASO, DT_FS):
    datos = leer_trayectoria(TRAYECTORIA, paso=PASO, dt_fs=DT_FS)
    _clave = (TRAYECTORIA, PASO, DT_FS)
print(f"{len(datos['V'])} frames, {len(datos['masas'])} átomos, dt = {datos['dt_fs']:.4f} fs ({datos['origen_dt']})")

## 4. VDOS y PR

In [ ]:
res = calcular(datos, inicio_ps=INICIO_PS, fin_ps=FIN_PS, seg_ps=SEG_PS, q=Q, con_masa=CON_MASA)
resumen(res)

In [ ]:
figura_vdos_pr(res, sombrear=SOMBREAR);

Comprobaciones: la temperatura debe estar estable en el tramo usado y la VDOS de las dos mitades debe coincidir.

In [ ]:
figura_diagnostico(res);